In [1]:
from ml_end_to_end.features import build_features
df = build_features()
print(df.shape)

(3000888, 16)


In [2]:
df_sorted = df.sort_values("date")
split_date = df_sorted["date"].quantile(0.8, interpolation="nearest")
train_part = df[df["date"] <= split_date]
valid_part = df[df["date"] > split_date]

print(train_part["date"].max(), "|", valid_part["date"].min())
print(len(train_part), len(valid_part))

2016-09-12 00:00:00 | 2016-09-13 00:00:00
2402136 598752


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = ["family", "store_nbr", "type", "city"]
num_cols = ["onpromotion", "cluster"]
bin_cols = ["is_weekend", "is_payday", "is_holiday", "is_event"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
], remainder="passthrough")

X_train = train_part[cat_cols + num_cols + bin_cols]
X_valid = valid_part[cat_cols + num_cols + bin_cols]

X_train_t = pre.fit_transform(X_train)   # fit + transform
X_valid_t = pre.transform(X_valid)        # SADECE transform

print(X_train_t.shape, X_valid_t.shape)

(2402136, 120) (598752, 120)


In [4]:
from sklearn.model_selection import KFold, TimeSeriesSplit

dates = df_sorted["date"].unique()

kf = KFold(n_splits=5)
tscv = TimeSeriesSplit(n_splits=5)

print("KFold ilk fold test tarihleri:", dates[list(kf.split(dates))[0][1]][:3], "...")
print("TimeSeriesSplit ilk fold test tarihleri:", dates[list(tscv.split(dates))[0][1]][:3], "...")

KFold ilk fold test tarihleri: <DatetimeArray>
['2013-01-01 00:00:00', '2013-01-02 00:00:00', '2013-01-03 00:00:00']
Length: 3, dtype: datetime64[us] ...
TimeSeriesSplit ilk fold test tarihleri: <DatetimeArray>
['2013-10-12 00:00:00', '2013-10-13 00:00:00', '2013-10-14 00:00:00']
Length: 3, dtype: datetime64[us] ...


In [5]:
train_idx, test_idx = list(kf.split(dates))[0]
print("KFold fold1 — eğitimin en son tarihi:", dates[train_idx].max())
print("KFold fold1 — testin en son tarihi:", dates[test_idx].max())

KFold fold1 — eğitimin en son tarihi: 2017-08-15 00:00:00
KFold fold1 — testin en son tarihi: 2013-12-03 00:00:00


## Bulgular
1. Zamana göre bölme (son %20) şart — rastgele bölme veya düz KFold,
   modelin gelecekten geçmişi tahmin etmesine izin veriyor (sızıntı).
2. KFold fold1'de doğrulandı: eğitim 2017'ye kadar uzanıyor, test 2013'te
   kalıyor — ters kronoloji.
3. Dönüştürücü (OneHotEncoder, StandardScaler) SADECE train_part'a fit
   edilmeli, valid_part'a sadece transform uygulanmalı.
4. Zaman serisinde doğru CV aracı TimeSeriesSplit — her fold'da eğitim
   her zaman test'ten önceki zamanı kapsıyor. Detaylı kullanım Perşembe.